## SITCOM-2124
This notebook will be the first step in the analysis we discussed during today's SPA meeting. Here is what I would do to implement it.\
\
First, we need to determine the multiple executions of `maintel/warmup_hexapod.py` over several days and their corresponding salIndex. There may be several ways to do it. Since we are looking for the implementation of a function, let’s try one approach. The query below uses the scriptState to report what happened to a script. You can look up ScriptState in the Script enumeration table (the code in GitHub may be easier to read) to determine what each value represents. 

```sql
SELECT "scriptSalIndex", "path", "salIndex", "scriptState"
FROM "efd"."autogen"."lsst.sal.ScriptQueue.logevent_script"
WHERE time > :dashboardTime:
AND time < :upperDashboardTime:
AND path =~ /warmup_hexapod/
```
\
You can try this out in Chronograf. The path `=~ /warmup_hexapod/` means that we are querying only rows where the path contains the `warmup_hexapod` string (this is a regex expression).
Now, you need to mine the script configuration to determine which hexapod was being warmed up by each script. This is another query: 

```sql
SELECT "salIndex", "blockId", "config" 
FROM "efd"."autogen"."lsst.sal.Script.command_configure" 
WHERE time > :dashboardTime: 
AND time < :upperDashboardTime: 
AND config =~ /hexapod\: m2/ 
```

This will query all scripts where the configuration contains the `‘hexapod: m2’` string. You can also try this out in Chronograf. However, be aware that Chronograf can be somewhat annoying with the
`:` symbol. \
\
Once you have both tables, you need to cross-match them to select only the scripts used to warm up M2Hex. This is the initial step.
Keep the `scriptState` in the first table because you might be able to use it to filter which scripts passed or failed later.
You can use the example in this notebook to learn how to do the query using `InfluxQL` in a notebook. This function will be an `async` function.

In [ ]:
day_start = 20250815
day_end = 20250831

path = 'warmup_hexapod'
config = 'hexapod: m2'
#sal_index = 2

In [ ]:
import pandas as pd
from astropy.time import Time

from lsst.summit.utils.efdUtils import getDayObsEndTime, getDayObsStartTime, makeEfdClient

efd_client = makeEfdClient()

In [ ]:
async def query_logevent_script(client, day_start, day_end, path, sal_index):

    """
    Query the logevent_script measurement for a specific day of observation.

    Parameters:
        client: EFD client, or USDF client
        day_start, day_end: datetime or DayObs
        path: str, match the script path
        sal_index: int, scriptSalIndex to filter

    Returns:
        pd.DataFrame with columns ['scriptSalIndex', 'path', 'salIndex', 'scriptState']
    """

    start_time = Time(getDayObsStartTime(day_start), scale = 'utc')
    end_time = Time(getDayObsEndTime(day_end), scale = 'utc')

    query = f'''
            SELECT "scriptSalIndex", "path", "salIndex", "scriptState"
            FROM "efd"."autogen"."lsst.sal.ScriptQueue.logevent_script" 
            WHERE time > '{start_time.isot}Z' 
            AND time < '{end_time.isot}Z' 
            AND path =~ /{path}/
            AND salIndex = {sal_index}
            '''
    df_scripts = await client.influx_client.query(query)

    return(df_scripts)

df_scripts = await query_logevent_script(efd_client,day_start, day_end, path, 1) # Not CamHex or M2Hex? 

df_scripts # Match time start of script with command_configure


For the easier data handling later, we'll save the index as the "time" column

In [ ]:
df_scripts = df_scripts.reset_index().rename(columns={"index": "time"})
df_scripts

If you want to check what the `scriptState` means, you can check the link: 

https://github.com/lsst-ts/ts_xml/blob/develop/python/lsst/ts/xml/enums/Script.py.

You can also print the specific state by running: 

```sql
from lsst.ts.xml.enums.Script import ScriptState
ScriptState(10).name
```

You can also print all script states by running: 

```sql
df_states = pd.DataFrame(
    [(state.value, state.name) for state in ScriptState],
    columns=["Number", "Name"]
)

df_states`
```

In [ ]:
async def query_command_configure(client, day_start, day_end, path):
    """
    Query the command_configure measurement for a specific configuration.

    Parameters:
        client: EFD client, or USDF client
        day_start, day_end: datetime or DayObs
        path: str, match the config column

    Returns:
        pd.DataFrame with columns ['salIndex', 'blockId', 'config', 'ScriptID']
    """

    start_time = Time(getDayObsStartTime(day_start), scale = 'utc')
    end_time = Time(getDayObsEndTime(day_end), scale = 'utc')

    query = f'''
            SELECT "salIndex", "blockId", "config", "ScriptID"
            FROM "efd"."autogen"."lsst.sal.Script.command_configure"
            WHERE time > '{start_time.isot}Z'
            AND time < '{end_time.isot}Z'
            AND config =~ /{config}/
            '''
    
    df_config = await client.influx_client.query(query)

    return(df_config)

df_config = await query_command_configure(efd_client, day_start, day_end, config)

df_config

In [ ]:
# Merge
df_m2hex = pd.merge(
    df_scripts,
    df_config,
    left_on="scriptSalIndex",
    right_on="salIndex",
    how="inner",  # only keep matches
    suffixes=("_script", "_config")
)

# Filter only warmup of M2 hexapod
df_m2hex = df_m2hex[df_m2hex["config"].str.contains("hexapod: m2")]

# Keep only relevant columns (use the names that exist after merge)
df_m2hex = df_m2hex[[
    "time","scriptSalIndex", "scriptState",
    "blockId", "config"
]]

df_m2hex#.head()
